# Swarm / Handoff

> Notebook generated from [`examples/patterns/08_swarm.py`](../../examples/patterns/08_swarm.py).

| Field | Value |
|------|-------|
| **Dataset** | ATIS (embedded, 6 flight queries) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** Trail of handoffs between agents (booking, info, support); JSON audit with allow-list and reason for each transfer.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Swarm / Decentralized Handoff — Customer support routing
=====================================================================
Pattern: SPEC-PAT-007 / prismal.agents.patterns.swarm

Dataset: ATIS (Airline Travel Information System) + Support Tickets
  • ATIS: ~5,871 utterances classified into 26 travel/support intents.
  • Reference: https://huggingface.co/datasets/tuetschek/atis
  • Why: Swarm is ideal for support systems where different
    specialized agents handle different types of queries. ATIS
    provides real user intents that map naturally to
    handoffs between specialized agents.

Pattern description:
  swarm_handoff transfers control between agents without a central supervisor:
  1. The current agent decides which specialized agent to delegate to.
  2. A HandoffRecord is recorded in state["metadata"]["handoff_history"].
  3. state["next_agent"] is updated to the target agent.
  4. The new agent can make another handoff or resolve the query.

Guarantees:
  - Immutability: the input state is not mutated.
  - Anti-loop: self-handoff rejected with ValueError.
  - Allow-listing: only agents in valid_targets are valid destinations.
  - Audit trail: each handoff records timestamp, reason and snapshot.

Usage:
    uv run python examples/patterns/08_swarm.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio
from typing import Any

from prismal.agents.patterns.swarm import (
    VALID_HANDOFF_TARGETS,
    swarm_handoff,
)

## Dataset: customer support tickets

In [ ]:
# Inspired by ATIS + Zendesk/Freshdesk support ticket categories.
SUPPORT_TICKETS = [
    {
        "id": "ST001",
        "channel": "email",
        "text": (
            "Hi, I've been waiting 3 days for my order #12345 and it hasn't "
            "arrived. The estimated delivery date was yesterday. Can you track my package?"
        ),
        "initial_agent": "researcher",  # starts at the general researcher
        "expected_path": ["researcher", "file_manager"],  # research → manage file
        "intent": "order_tracking",
    },
    {
        "id": "ST002",
        "channel": "chat",
        "text": (
            "I need help writing a complex SQL query that computes "
            "the customer retention rate by monthly cohort over the last "
            "24 months. I have a pandas DataFrame with columns: user_id, signup_date, last_order."
        ),
        "initial_agent": "researcher",
        "expected_path": ["researcher", "coder", "data_analyst"],
        "intent": "technical_data_analysis",
    },
    {
        "id": "ST003",
        "channel": "phone_transcript",
        "text": (
            "The Q3 sales report has an error in the Europe columns. "
            "The numbers do not match what we have in the CRM. I need someone "
            "to analyze the data and generate a corrected report."
        ),
        "initial_agent": "researcher",
        "expected_path": ["researcher", "data_analyst", "file_manager"],
        "intent": "data_analysis_report",
    },
    {
        "id": "ST004",
        "channel": "ticket",
        "text": (
            "We have a critical bug in production. The /api/payments endpoint "
            "is returning 500 errors at a 23% rate in the last 30 minutes. "
            "The logs show: KeyError 'transaction_id' in payment_processor.py:L145"
        ),
        "initial_agent": "researcher",
        "expected_path": ["researcher", "coder", "critic"],
        "intent": "critical_bug_production",
    },
    {
        "id": "ST005",
        "channel": "slack",
        "text": (
            "Can you help me plan the Q4 sprint? We have 8 features "
            "to implement, 3 developers and 6 weeks. I need to estimate "
            "effort and prioritize by business impact."
        ),
        "initial_agent": "researcher",
        "expected_path": ["researcher", "planner"],
        "intent": "sprint_planning",
    },
]

## Agent routing logic

In [ ]:
def classify_intent(text: str) -> str:
    """Simple keyword-based intent classifier.

    In production, this would be an LLM intent classifier
    or a trained text classification model.

    Args:
        text: Support ticket text.

    Returns:
        Classified intent.
    """
    text_lower = text.lower()

    if any(kw in text_lower for kw in ["bug", "error", "exception", "500", "crash", "traceback"]):
        return "technical_bug"
    if any(
        kw in text_lower for kw in ["sql", "pandas", "dataframe", "analysis", "analyze", "report"]
    ):
        return "data_analysis"
    if any(kw in text_lower for kw in ["order", "delivery", "package", "shipping", "tracking"]):
        return "order_support"
    if any(kw in text_lower for kw in ["plan", "sprint", "prioritize", "roadmap", "features"]):
        return "planning"
    if any(kw in text_lower for kw in ["code", "implement", "function", "api", "endpoint"]):
        return "coding"
    return "general"


def decide_handoff(current_agent: str, intent: str, state: dict) -> tuple[str, str] | None:
    """Decide whether to hand off and to which agent.

    Implements the swarm's decentralized routing logic.

    Args:
        current_agent: Currently active agent.
        intent: Classified intent of the ticket.
        state: Current agent state.

    Returns:
        Tuple (target_agent, reason) or None if there is no handoff.
    """
    handoff_history = state.get("metadata", {}).get("handoff_history", [])
    agents_visited = {h["to_agent"] for h in handoff_history}
    agents_visited.add(current_agent)

    # Routing table: intent → sequence of specialized agents
    routing_table: dict[str, list[str]] = {
        "technical_bug": ["coder", "critic"],
        "data_analysis": ["data_analyst", "file_manager"],
        "order_support": ["file_manager"],
        "planning": ["planner"],
        "coding": ["coder"],
        "general": [],
    }

    target_sequence = routing_table.get(intent, [])

    # Find the first unvisited agent in the sequence
    for target in target_sequence:
        if target not in agents_visited and target in VALID_HANDOFF_TARGETS:
            reasons = {
                "coder": "Query requires code generation or analysis",
                "critic": "Generated code needs quality review",
                "data_analyst": "Query requires data analysis and statistics",
                "file_manager": "Needs file or report management",
                "planner": "Query requires task and sprint planning",
                "rag_agent": "Needs information retrieval from knowledge base",
                "researcher": "Requires investigation and information gathering",
            }
            return target, reasons.get(target, f"Specialist in {target}")

    return None  # No further handoffs needed


async def process_ticket(ticket: dict) -> dict[str, Any]:
    """Process a support ticket via swarm handoffs.

    Args:
        ticket: Ticket with text, channel and expected intent.

    Returns:
        Final state with the full handoff history.
    """
    # Initial ticket state
    state: dict[str, Any] = {
        "messages": [],
        "current_agent": ticket["initial_agent"],
        "metadata": {
            "ticket_id": ticket["id"],
            "channel": ticket["channel"],
            "handoff_history": [],
        },
    }

    intent = classify_intent(ticket["text"])
    current_agent = ticket["initial_agent"]
    max_hops = 5  # anti-loop safety limit

    print(f"\n[{ticket['id']}] Channel: {ticket['channel']}")
    print(f"  Detected intent: {intent}")
    print(f"  Text: {ticket['text'][:80]}...")
    print("\n  Routing:")

    hop = 0
    while hop < max_hops:
        handoff_decision = decide_handoff(current_agent, intent, state)

        if handoff_decision is None:
            print(f"    ✓ {current_agent} resolves the ticket (no further handoffs)")
            break

        target_agent, reason = handoff_decision

        try:
            # Execute the handoff
            state = await swarm_handoff(
                current_agent=current_agent,
                target_agent=target_agent,
                state=state,
                reason=reason,
                valid_targets=VALID_HANDOFF_TARGETS,
            )

            print(f"    {current_agent} → {target_agent}  [{reason}]")
            current_agent = state.get("next_agent", target_agent)
            hop += 1

        except Exception as exc:
            print(f"    ✗ Handoff failed: {exc}")
            break

    return state


async def main() -> None:
    print("=" * 70)
    print("  Swarm / Handoff — Dataset: ATIS + support tickets")
    print("=" * 70)
    print(f"\n  Valid agents in the swarm: {sorted(VALID_HANDOFF_TARGETS)}")

    all_states = []

    for ticket in SUPPORT_TICKETS:
        state = await process_ticket(ticket)
        all_states.append((ticket, state))

    # Routing summary
    print("\n" + "═" * 70)
    print("  ROUTING SUMMARY")
    print("═" * 70)
    print(f"  {'Ticket':<8} {'Channel':<20} {'Intent':<25} {'Hops':>6}")
    print("  " + "─" * 60)

    for ticket, state in all_states:
        history = state.get("metadata", {}).get("handoff_history", [])
        intent = classify_intent(ticket["text"])
        print(f"  {ticket['id']:<8} {ticket['channel']:<20} {intent:<25} {len(history):>6}")

    # Audit trail analysis
    print("\n[Audit Trail — Handoff History of the last ticket]")
    _last_ticket, last_state = all_states[-1]
    history = last_state.get("metadata", {}).get("handoff_history", [])
    for i, record in enumerate(history, 1):
        print(f"  {i}. {record['from_agent']} → {record['to_agent']}")
        print(f"     Reason   : {record['reason']}")
        print(f"     Timestamp: {record['timestamp']}")

    # Demonstrate rejected self-handoff
    print("\n[Safety validation — self-handoff]")
    try:
        await swarm_handoff(
            current_agent="coder",
            target_agent="coder",
            state={"metadata": {}},
            reason="self-handoff test",
        )
    except (ValueError, Exception) as e:
        print(f"  ✓ Self-handoff correctly rejected: {type(e).__name__}")

    # Demonstrate allow-listing
    print("\n[Safety validation — disallowed target]")
    try:
        await swarm_handoff(
            current_agent="coder",
            target_agent="malicious_agent",
            state={"metadata": {}},
            reason="invalid target test",
            valid_targets=VALID_HANDOFF_TARGETS,
        )
    except (ValueError, Exception) as e:
        print(f"  ✓ Invalid target correctly rejected: {type(e).__name__}")


if __name__ == "__main__":
    asyncio.run(main())


---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()